# Investment Agent — Diagnostics Notebook

Test each pipeline component independently before running the full orchestrator.  
Run cells top-to-bottom. Each section is self-contained with clear pass/fail output.

**Pipeline:** Config → Data → Indicators → Fundamentals → Strategies → Moderation → Risk → Execution → Journal → Orchestrator

**Data Rationale:** See [docs/DATA_RATIONALE.md](../docs/DATA_RATIONALE.md) for why each data point is kept or removed from the pipeline.

---
## 0. Environment Setup

In [ ]:
import sys, os, json
from pathlib import Path
from datetime import datetime

# Ensure project root is on sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print(f"Project root : {PROJECT_ROOT}")
print(f"Python       : {sys.version}")
print(f"Timestamp    : {datetime.utcnow().isoformat()}Z")

---
## 1. Configuration & Environment Variables

In [ ]:
from src.utils.config import get_settings, Settings

settings = get_settings()

print("=== Trading Settings ===")
print(f"  Mode            : {settings.trading.get('mode', 'N/A')}")
print(f"  Base URL        : {settings.t212_base_url}")
print(f"  Cycle hours     : {settings.cycle_hours}")
print(f"  Cycle times UTC : {settings.cycle_times_utc}")
print(f"  Max positions   : {settings.max_positions}")
print(f"  Min position %  : {settings.min_position_pct}")
print(f"  Max position %  : {settings.max_position_pct}")
print(f"  Cash floor %    : {settings.cash_floor_pct}")
print(f"  Benchmark       : {settings.benchmark_ticker}")

print("\n=== Risk Settings ===")
print(f"  Max single stock : {settings.max_single_stock_pct}%")
print(f"  Max sector       : {settings.max_sector_pct}%")
print(f"  Max correlation  : {settings.max_correlation}")
print(f"  Cautious DD %    : {settings.cautious_drawdown_pct}")
print(f"  Halt DD %        : {settings.halt_drawdown_pct}")
print(f"  VIX high/extreme : {settings.vix_high} / {settings.vix_extreme}")

print("\n=== Strategy Settings ===")
print(f"  Momentum weight      : {settings.momentum_weight}")
print(f"  Mean reversion weight : {settings.mean_reversion_weight}")
print(f"  Factor weight        : {settings.factor_weight}")
print(f"  Min conviction       : {settings.min_conviction}")

print("\n=== Models ===")
print(f"  Strategy   : {settings.strategy_model}")
print(f"  Moderator 1: {settings.moderator_1_model}")
print(f"  Moderator 2: {settings.moderator_2_model}")

print("\n=== Cost Limits ===")
print(f"  Anthropic daily : \u00a3{settings.anthropic_daily_gbp}")
print(f"  OpenAI daily    : \u00a3{settings.openai_daily_gbp}")
print(f"  Google daily    : \u00a3{settings.google_daily_gbp}")
print(f"  Monthly total   : \u00a3{settings.total_monthly_gbp}")

print("\n=== Config: PASS ===")

In [ ]:
# Verify all required environment variables are set (mask values)
env_keys = [
    "T212_API_KEY", "T212_API_SECRET",
    "ANTHROPIC_API_KEY", "OPENAI_API_KEY", "GOOGLE_AI_API_KEY",
    "FINNHUB_API_KEY", "ALPHA_VANTAGE_API_KEY",
]

all_ok = True
for key in env_keys:
    val = os.getenv(key)
    if val:
        masked = val[:4] + "..." + val[-4:] if len(val) > 8 else "****"
        print(f"  {key:25s} = {masked}")
    else:
        print(f"  {key:25s} = MISSING")
        all_ok = False

env_status = "PASS" if all_ok else "WARN (some keys missing)"
print(f"\n=== Env Vars: {env_status} ===")

---
## 2. Database & Models

In [ ]:
from sqlalchemy import inspect as sa_inspect, text
from src.data.database import engine, get_session
from src.data.models import Base

Base.metadata.create_all(engine)

inspector = sa_inspect(engine)
table_names = inspector.get_table_names()
print(f"Database : {engine.url}")
print(f"Tables ({len(table_names)}):")

session = get_session()
try:
    for t in sorted(table_names):
        count = session.execute(text(f'SELECT COUNT(*) FROM "{t}"')).scalar()
        print(f"  {t:35s} {count:>6} rows")
finally:
    session.close()

print(f"\n=== Database: PASS ===")

---
## 3. State Machine

In [ ]:
from src.orchestrator.state_machine import StateMachine

sm = StateMachine()
state = sm.get_state()

print("=== System State ===")
for k, v in state.items():
    print(f"  {k:25s} : {v}")

assert state["state"] in {"ACTIVE", "CAUTIOUS", "HALTED"}, f"Invalid state: {state['state']}"
print(f"\nCurrent state : {state['state']}")
print(f"Is paused     : {state.get('paused', False)}")
print(f"\n=== State Machine: PASS ===")

---
## 4. Cost Tracker & Budget Status

In [ ]:
from src.utils.cost_tracker import (
    Provider, get_budget_status, get_degradation_level,
    get_cost_summary, get_daily_spend, get_monthly_spend,
)

print("=== Budget Status ===")
for provider in Provider:
    bs = get_budget_status(provider.value)
    print(f"\n  {provider.value.upper()}:")
    print(f"    Daily  : \u00a3{bs.daily_spent_gbp:.4f} / \u00a3{bs.daily_limit_gbp:.2f}  ({bs.daily_pct_used:.1f}%)")
    print(f"    Monthly: \u00a3{bs.monthly_spent_gbp:.4f} / \u00a3{bs.monthly_limit_gbp:.2f}  ({bs.monthly_pct_used:.1f}%)")
    print(f"    Over daily: {bs.is_over_daily}  |  Over monthly: {bs.is_over_monthly}")

deg = get_degradation_level()
print(f"\nDegradation level : {deg.value}")
print(f"Total today       : \u00a3{get_daily_spend():.4f}")
print(f"Total this month  : \u00a3{get_monthly_spend():.4f}")

summary = get_cost_summary(days=7)
print(f"\n7-day summary: {json.dumps(summary, indent=2)}")

print(f"\n=== Cost Tracker: PASS ===")

---
## 5. Market Data — yfinance (OHLCV + Indicators)

In [ ]:
from src.agents.market_data.data_fetcher import DataFetcher

fetcher = DataFetcher()

TEST_TICKER = "AAPL"
print(f"Fetching 1-year OHLCV for {TEST_TICKER}...")
df = fetcher.get_ohlcv(TEST_TICKER, period="1y")
assert not df.empty, "OHLCV data is empty!"
print(f"  Rows    : {len(df)}")
print(f"  Columns : {list(df.columns)}")
print(f"  Range   : {df.index[0].date()} to {df.index[-1].date()}")
print(f"  Latest close: ${float(df['Close'].iloc[-1]):.2f}")

print(f"\n=== yfinance OHLCV: PASS ===")

In [ ]:
from src.agents.market_data.indicators import calculate_indicators, calculate_relative_strength

print(f"Calculating indicators for {TEST_TICKER}...")
indicators = calculate_indicators(df)

assert "error" not in indicators, f"Indicator error: {indicators.get('error')}"

# Verify only the 8 expected fields are present (see docs/DATA_RATIONALE.md)
EXPECTED_INDICATOR_KEYS = {
    "current_price", "rsi_14", "macd_histogram",
    "macd_bullish_crossover", "macd_bearish_crossover",
    "below_lower_bb", "above_50ma", "ma_20",
}
actual_keys = set(indicators.keys())
assert actual_keys == EXPECTED_INDICATOR_KEYS, (
    f"Unexpected indicator keys.\n"
    f"  Extra: {actual_keys - EXPECTED_INDICATOR_KEYS}\n"
    f"  Missing: {EXPECTED_INDICATOR_KEYS - actual_keys}"
)

print(f"\n=== Technical Indicators ({len(indicators)} fields) ===")
for k, v in indicators.items():
    if isinstance(v, float):
        print(f"  {k:25s} : {v:.4f}")
    else:
        print(f"  {k:25s} : {v}")

print(f"\nFetching benchmark data...")
bench_df = fetcher.get_benchmark_data()
assert not bench_df.empty, "Benchmark data is empty!"
rs = calculate_relative_strength(df, bench_df)
print(f"  {TEST_TICKER} relative strength (6m) vs S&P 500: {rs:.4f}")

print(f"\n=== Indicators: PASS ===")

---
## 6. Fundamentals (yfinance)

In [ ]:
from src.agents.market_data.fundamentals import get_fundamentals

print(f"Fetching fundamentals for {TEST_TICKER}...")
fundamentals = get_fundamentals(TEST_TICKER)

assert "error" not in fundamentals, f"Fundamentals error: {fundamentals.get('error')}"

# Verify only the 9 expected fields are present (see docs/DATA_RATIONALE.md)
EXPECTED_FUNDAMENTAL_KEYS = {
    "trailing_pe", "pb_ratio", "roe", "profit_margin",
    "debt_equity", "earnings_growth", "earnings_momentum_qoq",
    "sector", "market_cap",
}
actual_keys = set(fundamentals.keys())
assert actual_keys == EXPECTED_FUNDAMENTAL_KEYS, (
    f"Unexpected fundamental keys.\n"
    f"  Extra: {actual_keys - EXPECTED_FUNDAMENTAL_KEYS}\n"
    f"  Missing: {EXPECTED_FUNDAMENTAL_KEYS - actual_keys}"
)

print(f"\n=== Fundamentals ({len(fundamentals)} fields) ===")
for k, v in fundamentals.items():
    if isinstance(v, float) and v is not None:
        print(f"  {k:25s} : {v:.4f}")
    else:
        print(f"  {k:25s} : {v}")

print(f"\n=== Fundamentals: PASS ===")

---
## 7. Macro Data & Market Regime

In [ ]:
print("Fetching macro data (VIX, S&P 500 vs 200-day MA)...")
macro = fetcher.get_macro_data()

print(f"\n=== Macro Data ===")
for k, v in macro.items():
    if isinstance(v, float):
        print(f"  {k:25s} : {v:.4f}")
    else:
        print(f"  {k:25s} : {v}")

# Verify yield spread fields were removed
assert "yield_spread_10y_short" not in macro, "yield_spread should be removed (see DATA_RATIONALE.md)"
assert "ten_year_yield" not in macro, "ten_year_yield should be removed"
assert "short_yield" not in macro, "short_yield should be removed"

print(f"\nMarket regime : {macro.get('market_regime', 'UNKNOWN')}")
print(f"VIX           : {macro.get('vix', 'N/A')}")

print(f"\n=== Macro Data: PASS ===")

---
## 8. Finnhub API (Analyst Recommendations + Insider Sentiment)

In [ ]:
print(f"Testing Finnhub API for {TEST_TICKER}...")

try:
    finnhub = fetcher.finnhub

    print("\n--- Analyst Recommendations ---")
    recs = finnhub.get_analyst_recommendations(TEST_TICKER)
    for k, v in recs.items():
        print(f"  {k:20s} : {v}")

    print("\n--- Insider Sentiment ---")
    insider = finnhub.get_insider_sentiment(TEST_TICKER)
    for k, v in insider.items():
        print(f"  {k:20s} : {v}")

    print("\n--- Combined Analyst Data ---")
    analyst_data = finnhub.get_analyst_data(TEST_TICKER)
    print(f"  Keys: {list(analyst_data.keys())}")

    finnhub_status = "PASS"
except Exception as e:
    print(f"  ERROR: {e}")
    finnhub_status = f"FAIL ({e})"

print(f"\n=== Finnhub: {finnhub_status} ===")

---
## 9. Alpha Vantage API (News Sentiment)

**Note:** Free tier = 25 requests/day. This section uses 2 calls.

In [ ]:
print("Testing Alpha Vantage news sentiment API...\n")

try:
    av = fetcher.alpha_vantage

    print("--- Broad Market Sentiment ---")
    broad = av.get_broad_market_sentiment()
    if "error" in broad:
        print(f"  Error: {broad['error']}")
    else:
        print(f"  Total articles   : {broad.get('total_articles', 0)}")
        print(f"  Avg sentiment    : {broad.get('average_sentiment', 0):.4f}")
        print(f"  Bullish articles : {broad.get('bullish_articles', 0)}")
        print(f"  Bearish articles : {broad.get('bearish_articles', 0)}")
        broad_articles = broad.get("articles", [])
        if broad_articles:
            print(f"  Top headlines:")
            from src.agents.market_data.alpha_vantage_client import AlphaVantageClient
            for line in AlphaVantageClient._summarize_articles(broad_articles, max_articles=5).split('\n'):
                print(f"    {line}")

    print(f"\n--- Ticker Sentiment ({TEST_TICKER}) ---")
    ticker_sent = av.get_ticker_news_summary(tickers=TEST_TICKER, limit=10)
    if "error" in ticker_sent:
        print(f"  Error: {ticker_sent['error']}")
    else:
        print(f"  Total articles   : {ticker_sent.get('total_articles', 0)}")
        print(f"  Avg sentiment    : {ticker_sent.get('average_sentiment', 0):.4f}")
        print(f"  Bullish articles : {ticker_sent.get('bullish_articles', 0)}")
        print(f"  Bearish articles : {ticker_sent.get('bearish_articles', 0)}")
        summary_text = ticker_sent.get('top_articles_summary', '')
        if summary_text and summary_text != "No recent articles found.":
            print(f"  Top 10 articles (fed to LLM):")
            for line in summary_text.split('\n'):
                print(f"    {line}")
        else:
            print(f"  WARNING: No articles returned — LLM will not receive news context")

    # Build the same news_sentiment string the orchestrator builds (for use in cell 12)
    news_parts = []
    if ticker_sent and "error" not in ticker_sent:
        news_parts.append(f"### Ticker News ({ticker_sent.get('tickers_queried', 'N/A')})")
        news_parts.append(f"Articles: {ticker_sent.get('total_articles', 0)} | "
                          f"Avg sentiment: {ticker_sent.get('average_sentiment', 0):.4f} | "
                          f"Bullish: {ticker_sent.get('bullish_articles', 0)} | "
                          f"Bearish: {ticker_sent.get('bearish_articles', 0)}")
        summary = ticker_sent.get("top_articles_summary", "")
        if summary:
            news_parts.append(summary)
    if broad and "error" not in broad:
        news_parts.append(f"\n### Broad Market Sentiment")
        news_parts.append(f"Articles: {broad.get('total_articles', 0)} | "
                          f"Avg sentiment: {broad.get('average_sentiment', 0):.4f} | "
                          f"Bullish: {broad.get('bullish_articles', 0)} | "
                          f"Bearish: {broad.get('bearish_articles', 0)}")
        broad_arts = broad.get("articles", [])
        if broad_arts:
            news_parts.append(AlphaVantageClient._summarize_articles(broad_arts, max_articles=10))

    diag_news_sentiment = "\n".join(news_parts)[:3000] if news_parts else "News sentiment data unavailable."

    print(f"\n--- News Sentiment String for LLM (preview) ---")
    print(diag_news_sentiment[:500])
    if len(diag_news_sentiment) > 500:
        print(f"  ... ({len(diag_news_sentiment)} chars total)")

    av_status = "PASS"
except Exception as e:
    print(f"  ERROR: {e}")
    diag_news_sentiment = "News sentiment data unavailable."
    av_status = f"FAIL ({e})"

print(f"\n=== Alpha Vantage: {av_status} ===")

---
## 10. Full Stock Analysis (Combined Data Pipeline)

In [ ]:
print(f"Running full stock analysis for {TEST_TICKER}...")
print("(Combines: OHLCV + indicators + relative strength + fundamentals + Finnhub)\n")

analysis = fetcher.get_stock_analysis(TEST_TICKER)

print(f"Top-level keys: {list(analysis.keys())}")

ind_keys = list(analysis.get("indicators", {}).keys())
fund_keys = list(analysis.get("fundamentals", {}).keys())
print(f"\nIndicators ({len(ind_keys)})  : {ind_keys}")
print(f"Fundamentals ({len(fund_keys)}): {fund_keys}")
print(f"Relative strength 6m : {analysis.get('relative_strength_6m', 'N/A')}")
print(f"Analyst data keys    : {list(analysis.get('analyst_data', {}).keys())}")

assert "indicators" in analysis
assert "fundamentals" in analysis

# Verify cleaned data shapes
ind = analysis.get("indicators", {})
if "error" not in ind:
    assert len(ind) == 8, f"Expected 8 indicator fields, got {len(ind)}: {list(ind.keys())}"
fund = analysis.get("fundamentals", {})
if "error" not in fund:
    assert len(fund) == 9, f"Expected 9 fundamental fields, got {len(fund)}: {list(fund.keys())}"

print(f"\n=== Full Stock Analysis: PASS ===")

---
## 11. Sub-Strategies (Momentum / Mean Reversion / Factor)

In [ ]:
from src.agents.strategy.momentum import evaluate_momentum
from src.agents.strategy.mean_reversion import evaluate_mean_reversion
from src.agents.strategy.factor import calculate_factor_score, rank_by_factor

ind = analysis.get("indicators", {})
fund = analysis.get("fundamentals", {})
rs_val = analysis.get("relative_strength_6m")

print("=== Momentum Strategy ===")
mom = evaluate_momentum(TEST_TICKER, ind, rs_val, current_holding=False)
print(f"  Action    : {mom.action}")
print(f"  Score     : {mom.score:.0f}/100")
print(f"  Reasoning : {mom.reasoning}")

print("\n=== Mean Reversion Strategy ===")
mr = evaluate_mean_reversion(TEST_TICKER, ind, fund, current_holding=False)
print(f"  Action    : {mr.action}")
print(f"  Score     : {mr.score:.0f}/100")
print(f"  Reasoning : {mr.reasoning}")

print("\n=== Factor Strategy ===")
factor = calculate_factor_score(TEST_TICKER, fund, ind, rs_val)
print(f"  Composite : {factor.composite_score:.0f}/100")
print(f"  Value     : {factor.value_score:.0f}")
print(f"  Quality   : {factor.quality_score:.0f}")
print(f"  Momentum  : {factor.momentum_score:.0f}")
print(f"  Reasoning : {factor.reasoning}")

print(f"\n=== Sub-Strategies: PASS ===")

In [ ]:
from src.agents.strategy.engine import StrategyEngine

MULTI_TICKERS = ["AAPL", "MSFT", "GOOGL"]
print(f"Testing sub-strategies on: {MULTI_TICKERS}")
print("(Fetching data — may take a moment...)\n")

stocks_data = []
for ticker in MULTI_TICKERS:
    try:
        data = fetcher.get_stock_analysis(ticker)
        data["ticker"] = ticker
        stocks_data.append(data)
        print(f"  {ticker}: OK (price=${data.get('indicators', {}).get('current_price', 'N/A')})")
    except Exception as e:
        print(f"  {ticker}: ERROR - {e}")
        stocks_data.append({"ticker": ticker, "indicators": {}, "fundamentals": {}})

engine = StrategyEngine()
sub_results = engine.run_sub_strategies(stocks_data, existing_positions=set())

print(f"\nMomentum signals : {len(sub_results['momentum'])}")
for s in sub_results["momentum"]:
    print(f"  {s.ticker}: {s.action} (score={s.score:.0f})")

print(f"\nMean reversion signals : {len(sub_results['mean_reversion'])}")
for s in sub_results["mean_reversion"]:
    print(f"  {s.ticker}: {s.action} (score={s.score:.0f})")

print(f"\nTop factor scores : {len(sub_results.get('top_factor', []))}")
for s in sub_results.get("top_factor", []):
    print(f"  {s.ticker}: composite={s.composite_score:.0f} (V={s.value_score:.0f} Q={s.quality_score:.0f} M={s.momentum_score:.0f})")

print(f"\n=== Multi-Ticker Sub-Strategies: PASS ===")

---
## 12. Claude Strategy Synthesis (Anthropic API)

**Cost:** ~$0.01-0.03 per call.

In [ ]:
print("Testing Claude strategy synthesis...")
print("(This makes a real Anthropic API call)\n")

# Use real news sentiment from cell 9 if available
_news_for_claude = globals().get("diag_news_sentiment", "News sentiment data unavailable.")
print(f"News sentiment source: {'Real Alpha Vantage data' if _news_for_claude != 'News sentiment data unavailable.' else 'Placeholder (AV unavailable)'}")
print(f"News sentiment length: {len(_news_for_claude)} chars\n")

try:
    portfolio_state = json.dumps({
        "cash": 10000.0, "total_value": 10000.0,
        "invested": 0.0, "positions": [], "num_positions": 0,
    })

    synth_result = engine.synthesize_with_claude(
        sub_strategy_results=sub_results,
        portfolio_state=portfolio_state,
        market_regime=macro.get("market_regime", "SIDEWAYS"),
        analyst_data="{}",
        news_sentiment=_news_for_claude,
        system_state="ACTIVE",
        vix=macro.get("vix"),
        cash_pct=100.0,
        num_positions=0,
        cycle_id="diag_test",
    )

    if "error" in synth_result and not synth_result.get("decisions"):
        print(f"  Strategy error: {synth_result['error']}")
        claude_status = f"WARN ({synth_result['error']})"
    else:
        decisions = synth_result.get("decisions", [])
        print(f"  Market assessment : {synth_result.get('market_assessment', 'N/A')[:120]}")
        print(f"  Decisions         : {len(decisions)}")
        for d in decisions[:5]:
            print(f"    {d.get('ticker')}: {d.get('action')} @ {d.get('target_allocation_pct', 0):.1f}%  conviction={d.get('conviction', 0)}")
            news_sum = d.get('news_sentiment_summary', '')
            if news_sum:
                print(f"      News context: {news_sum}")
        claude_status = "PASS"

except Exception as e:
    print(f"  ERROR: {e}")
    claude_status = f"FAIL ({e})"

print(f"\n=== Claude Synthesis: {claude_status} ===")

---
## 13. Moderation Panel (GPT-4o + Gemini)

**Cost:** ~$0.006 (OpenAI + Google combined).

In [ ]:
from src.agents.moderation import openai_mod, gemini_mod

sample_trade = {
    "ticker": "AAPL_US_EQ",
    "action": "BUY",
    "target_allocation_pct": 8.0,
    "conviction": 72,
    "primary_strategy": "momentum",
    "reasoning": "Strong momentum: above 50-day MA, RSI in sweet spot, positive MACD.",
}
sample_portfolio = json.dumps({"cash": 10000, "total_value": 10000, "positions": []})

# Build rich market_context dict (same structure the orchestrator builds)
sample_market_context = {
    "indicators": ind,
    "fundamentals": fund,
    "macro": {
        "vix": macro.get("vix"),
        "market_regime": macro.get("market_regime", "SIDEWAYS"),
        "sp500_above_200ma": macro.get("sp500_above_200ma"),
    },
    "sub_strategies": {
        "momentum": {"action": mom.action, "score": mom.score, "reasoning": mom.reasoning},
        "mean_reversion": {"action": mr.action, "score": mr.score, "reasoning": mr.reasoning},
        "factor": {
            "composite_score": factor.composite_score,
            "value_score": factor.value_score,
            "quality_score": factor.quality_score,
            "momentum_score": factor.momentum_score,
            "reasoning": factor.reasoning,
        },
    },
    "analyst_data": analysis.get("analyst_data", {}),
    "news_sentiment": globals().get("diag_news_sentiment", ""),
}

# Preview the formatted context
from src.agents.moderation.context import format_market_context
formatted_ctx = format_market_context(sample_market_context)
print(f"--- Market Context for Moderators ({len(formatted_ctx)} chars) ---")
print(formatted_ctx[:600])
if len(formatted_ctx) > 600:
    print(f"  ... ({len(formatted_ctx)} chars total)")

# GPT-4o moderator
print("\n=== GPT-4o Moderator ===")
try:
    gpt_result = openai_mod.review_trade(sample_trade, sample_portfolio, sample_market_context, "diag_test")
    for k, v in gpt_result.items():
        print(f"  {k:20s} : {v}")
    gpt_status = "PASS" if gpt_result.get("available") else f"WARN ({gpt_result.get('reasoning')})"
except Exception as e:
    print(f"  ERROR: {e}")
    gpt_status = f"FAIL ({e})"

# Gemini moderator
print(f"\n=== Gemini Moderator ===")
try:
    gem_result = gemini_mod.review_trade(sample_trade, sample_portfolio, sample_market_context, "diag_test")
    for k, v in gem_result.items():
        print(f"  {k:20s} : {v}")
    gem_status = "PASS" if gem_result.get("available") else f"WARN ({gem_result.get('reasoning')})"
except Exception as e:
    print(f"  ERROR: {e}")
    gem_status = f"FAIL ({e})"

print(f"\n=== GPT-4o: {gpt_status} ===")
print(f"=== Gemini: {gem_status} ===")

In [ ]:
from src.agents.moderation.panel import ModerationPanel

print("=== Full Moderation Panel ===")
try:
    panel = ModerationPanel()
    mod_result = panel.review_trade(
        trade_proposal=sample_trade,
        portfolio_context=sample_portfolio,
        market_context=sample_market_context,
        conviction=72,
        cycle_id="diag_test",
    )
    print(f"  Consensus            : {mod_result.consensus}")
    print(f"  Strategy verdict     : {mod_result.strategy_verdict}")
    print(f"  Moderators available : {mod_result.moderators_available}")
    print(f"  Caution flag         : {mod_result.caution_flag}")
    if mod_result.gpt4o_verdict:
        print(f"  GPT-4o verdict       : {mod_result.gpt4o_verdict.get('verdict', 'N/A')}")
        print(f"  GPT-4o reasoning     : {mod_result.gpt4o_verdict.get('reasoning', 'N/A')}")
        risk_flags = mod_result.gpt4o_verdict.get('risk_flags', [])
        if risk_flags:
            print(f"  GPT-4o risk flags    : {risk_flags}")
    if mod_result.gemini_verdict:
        print(f"  Gemini verdict       : {mod_result.gemini_verdict.get('verdict', 'N/A')}")
        print(f"  Gemini assessment    : {mod_result.gemini_verdict.get('assessment', 'N/A')}")
        print(f"  Gemini scores        : growth={mod_result.gemini_verdict.get('growth_score', 'N/A')} "
              f"risk={mod_result.gemini_verdict.get('risk_score', 'N/A')} "
              f"confidence={mod_result.gemini_verdict.get('confidence_score', 'N/A')}")
    panel_status = "PASS"
except Exception as e:
    print(f"  ERROR: {e}")
    panel_status = f"FAIL ({e})"

print(f"\n=== Moderation Panel: {panel_status} ===")

---
## 14. Risk Manager

In [ ]:
from src.agents.risk.risk_manager import RiskManager

rm = RiskManager()
vix_val = macro.get("vix")

print("=== Individual Risk Rule Checks ===\n")

rules = [
    ("Max single stock", rm.check_max_single_stock("AAPL_US_EQ", 10.0, {})),
    ("Max sector",       rm.check_max_sector("AAPL_US_EQ", "Technology", 10.0, {"Technology": 20.0})),
    ("VIX limit",        rm.check_vix_limit(vix_val, 10.0)),
    ("Cash floor",       rm.check_cash_floor(100.0, 10.0)),
    ("Daily loss halt",  rm.check_daily_loss_halt(0.0, None)),
    ("Drawdown",         rm.check_drawdown(10000, 10500)),
    ("Correlation",      rm.check_correlation({})),
    ("Min positions",    rm.check_min_positions(3, "SELL")),
    ("Cautious state",   rm.check_cautious_state("ACTIVE", "BUY", 10.0)),
]

for name, r in rules:
    icon = "PASS" if r.passed else "FAIL"
    print(f"  [{icon:4s}] {name:20s} : {r.message}")

print("\n=== Full Trade Evaluation ===")
verdict = rm.evaluate_trade(
    ticker="AAPL_US_EQ",
    action="BUY",
    proposed_allocation_pct=8.0,
    sector="Technology",
    current_portfolio={},
    sector_allocations={},
    portfolio_returns={},
    current_value=10000,
    peak_value=10000,
    cash_pct=100.0,
    vix=vix_val,
    daily_pnl_pct=0.0,
    daily_loss_halt_until=None,
    num_positions=0,
    system_state="ACTIVE",
    cycle_id="diag_test",
)
print(f"  Verdict              : {verdict.verdict}")
print(f"  Proposed allocation  : {verdict.proposed_allocation_pct:.1f}%")
print(f"  Adjusted allocation  : {verdict.adjusted_allocation_pct}")
print(f"  Rules checked        : {verdict.rules_checked}")
print(f"  Triggered rules      : {verdict.triggered_rules}")
print(f"  Reasoning            : {verdict.reasoning}")

print(f"\n=== Risk Manager: PASS ===")

---
## 15. Trading 212 Client (Practice Mode)

In [ ]:
from src.agents.execution.t212_client import T212Client

print("Testing Trading 212 API (Practice mode)...\n")

try:
    t212 = T212Client()

    print("--- Account Cash ---")
    cash = t212.get_cash()
    print(f"  {json.dumps(cash, indent=2)}")

    print("\n--- Account Info ---")
    info = t212.get_account_info()
    print(f"  {json.dumps(info, indent=2)}")

    print("\n--- Portfolio Positions ---")
    portfolio = t212.get_portfolio()
    print(f"  Positions: {len(portfolio)}")
    for pos in portfolio[:5]:
        print(f"    {pos.get('ticker', 'N/A')}: qty={pos.get('quantity', 0)} @ {pos.get('currentPrice', 0)}")

    t212.close()
    t212_status = "PASS"

except Exception as e:
    print(f"  ERROR: {e}")
    print("  (Expected if T212 API keys are not configured or demo server is down)")
    t212_status = f"WARN ({type(e).__name__})"

print(f"\n=== T212 Client: {t212_status} ===")

---
## 16. Order Manager (Dry Run)

In [ ]:
from src.agents.execution.order_manager import OrderManager
from src.agents.execution.t212_client import calculate_quantity

print("=== Order Quantity Calculation ===")
qty = calculate_quantity(target_amount=800.0, price=195.50)
print(f"  Target amount : \u00a3800.00")
print(f"  Price         : $195.50")
print(f"  Calculated qty: {qty}")
assert qty > 0, "Quantity should be positive"

print("\n=== Dry Run Order Execution ===")
try:
    om = OrderManager(dry_run=True)
    exec_result = om.execute_market_order(
        ticker="AAPL_US_EQ",
        action="BUY",
        target_amount_gbp=800.0,
        current_price=195.50,
        strategy="momentum",
        conviction=75,
        moderation_result="APPROVED",
        risk_result="APPROVE",
    )
    print(f"  Result: {json.dumps(exec_result, indent=2)}")
    assert exec_result["status"] in ("dry_run", "skipped"), f"Unexpected: {exec_result['status']}"
    om_status = "PASS"
except Exception as e:
    print(f"  ERROR: {e}")
    om_status = f"FAIL ({e})"

print(f"\n=== Order Manager: {om_status} ===")

---
## 17. Trade Journal Generation

In [ ]:
from src.agents.reporting.journal import generate_trade_journal

print("Testing trade journal generation...\n")

try:
    journal_path = generate_trade_journal(
        action="BUY",
        ticker="AAPL_US_EQ",
        shares=4.09,
        price=195.50,
        value_gbp=799.30,
        weight_pct=8.0,
        conviction=75,
        strategy="momentum",
        reasoning="Strong momentum: above 50-day MA, RSI in sweet spot, positive MACD.",
        growth_potential="MEDIUM",
        risk_level="LOW",
        catalysts=["iPhone cycle", "Services growth", "AI integration"],
        risks=["China exposure", "Regulatory risk", "Hardware saturation"],
        exit_conditions="Stop-loss at -8% or below 50-day MA for 3 consecutive days.",
        upside_target_pct=15.0,
        stop_loss_pct=8.0,
        expected_holding_period="3-6 months",
        market_regime=macro.get("market_regime", "SIDEWAYS"),
        vix=macro.get("vix"),
        sp500_trend=str(macro.get("sp500_pct_above_200ma", "N/A")),
        news_sentiment_overall="Mildly bullish",
        finnhub_data={"analyst_recommendations": {"consensus": "BUY", "total_analysts": 40}, "insider_sentiment": {"mspr": 0.5}},
        alpha_vantage_data={"average_sentiment": 0.15, "bullish_articles": 30, "bearish_articles": 8, "total_articles": 50},
        moderation_results={"consensus": "APPROVED", "strategy_verdict": "AGREE", "moderators_available": 2, "gpt4o_verdict": {"verdict": "AGREE"}, "gemini_verdict": {"verdict": "AGREE", "growth_score": 7, "risk_score": 4, "confidence_score": 6}},
        risk_verdict={"verdict": "APPROVE", "rules_checked": ["max_single_stock", "cash_floor", "vix_limit"], "triggered_rules": [], "reasoning": "All risk checks passed"},
        indicators=ind,
        fundamentals=fund,
        portfolio_state={"total_value": 10000, "cash": 9200, "invested": 800, "num_positions": 1, "total_return_pct": 0, "alpha_pct": 0, "positions": []},
    )

    print(f"  Journal written to: {journal_path}")
    assert os.path.exists(journal_path), "Journal file not created!"
    size = os.path.getsize(journal_path)
    print(f"  File size: {size:,} bytes")
    journal_status = "PASS"

except Exception as e:
    print(f"  ERROR: {e}")
    import traceback; traceback.print_exc()
    journal_status = f"FAIL ({e})"

print(f"\n=== Trade Journal: {journal_status} ===")

---
## 18. Full Orchestrator Dry Run

**Cost:** ~$0.03-0.05 (Anthropic + OpenAI + Google combined).

In [ ]:
from src.orchestrator.main import Orchestrator

print("Running FULL orchestrator cycle (dry_run=True)...")
print("This tests the complete pipeline end-to-end.\n")

try:
    orch = Orchestrator(dry_run=True)
    cycle_result = orch.run_cycle()

    print(f"Cycle ID     : {cycle_result.get('cycle_id', 'N/A')}")
    print(f"Status       : {cycle_result.get('status', 'N/A')}")
    print(f"Trades       : {cycle_result.get('num_trades', 0)}")
    print(f"Errors       : {cycle_result.get('errors', [])}")

    if cycle_result.get("trades"):
        print("\n--- Trade Details ---")
        for t in cycle_result["trades"]:
            print(f"  {t['ticker']}: {t['action']} @ {t['allocation_pct']:.1f}% | mod={t['moderation']} risk={t['risk']}")

    cost = cycle_result.get("cost_summary", {})
    if cost:
        print(f"\n--- Cost Summary (today) ---")
        for k, v in cost.items():
            print(f"  {k:15s} : \u00a3{v:.4f}")

    orch.close()
    orch_status = "PASS" if cycle_result.get("status") == "completed" else f"WARN ({cycle_result.get('status')})"

except Exception as e:
    print(f"  ERROR: {e}")
    import traceback; traceback.print_exc()
    orch_status = f"FAIL ({e})"

print(f"\n=== Orchestrator Dry Run: {orch_status} ===")

---
## 19. Post-Run Database Inspection

In [ ]:
from sqlalchemy import text
from src.data.database import get_session

session = get_session()
print("=== Recent Database Activity ===\n")

queries = {
    "Strategy decisions (last 5)": "SELECT timestamp, cycle_id, ticker, action, conviction, target_allocation_pct FROM strategy_decisions ORDER BY timestamp DESC LIMIT 5",
    "Moderation logs (last 5)": "SELECT timestamp, cycle_id, ticker, moderator, verdict, consensus FROM moderation_logs ORDER BY timestamp DESC LIMIT 5",
    "Risk decisions (last 5)": "SELECT timestamp, cycle_id, ticker, proposed_action, verdict, reasoning FROM risk_decisions ORDER BY timestamp DESC LIMIT 5",
    "Orders (last 5)": "SELECT timestamp, ticker, action, quantity, price, status, strategy FROM orders ORDER BY timestamp DESC LIMIT 5",
    "Cost log (last 10)": "SELECT timestamp, provider, model, input_tokens, output_tokens, cost_gbp, purpose FROM cost_logs ORDER BY timestamp DESC LIMIT 10",
    "API logs (last 5)": "SELECT timestamp, service, method, endpoint, status_code, duration_ms FROM api_logs ORDER BY timestamp DESC LIMIT 5",
}

try:
    for title, sql in queries.items():
        print(f"--- {title} ---")
        try:
            rows = session.execute(text(sql)).fetchall()
            if rows:
                for row in rows:
                    print(f"  {row}")
            else:
                print("  (no rows)")
        except Exception as e:
            print(f"  Query error: {e}")
        print()
finally:
    session.close()

print("=== Database Inspection: DONE ===")

---
## 20. Summary Report

In [ ]:
print("=" * 60)
print("  DIAGNOSTICS SUMMARY")
print("=" * 60)
print(f"  Timestamp : {datetime.utcnow().isoformat()}Z")
print("-" * 60)

results = {
    "1.  Configuration"      : "PASS",
    "2.  Database"           : "PASS",
    "3.  State Machine"      : "PASS",
    "4.  Cost Tracker"       : "PASS",
    "5.  yfinance OHLCV"    : "PASS",
    "6.  Indicators"         : "PASS",
    "7.  Fundamentals"       : "PASS",
    "8.  Macro Data"         : "PASS",
    "9.  Finnhub API"        : globals().get('finnhub_status', 'NOT RUN'),
    "10. Alpha Vantage API"  : globals().get('av_status', 'NOT RUN'),
    "11. Sub-Strategies"     : "PASS",
    "12. Claude Synthesis"   : globals().get('claude_status', 'NOT RUN'),
    "13a. GPT-4o Moderator" : globals().get('gpt_status', 'NOT RUN'),
    "13b. Gemini Moderator" : globals().get('gem_status', 'NOT RUN'),
    "13c. Moderation Panel" : globals().get('panel_status', 'NOT RUN'),
    "14. Risk Manager"       : "PASS",
    "15. T212 Client"        : globals().get('t212_status', 'NOT RUN'),
    "16. Order Manager"      : globals().get('om_status', 'NOT RUN'),
    "17. Trade Journal"      : globals().get('journal_status', 'NOT RUN'),
    "18. Orchestrator"       : globals().get('orch_status', 'NOT RUN'),
}

pass_count = sum(1 for v in results.values() if v == "PASS")
warn_count = sum(1 for v in results.values() if v.startswith("WARN"))
fail_count = sum(1 for v in results.values() if v.startswith("FAIL"))
not_run    = sum(1 for v in results.values() if v == "NOT RUN")

for name, result in results.items():
    if result == "PASS":
        icon = "[OK]"
    elif result.startswith("WARN"):
        icon = "[!!]"
    elif result.startswith("FAIL"):
        icon = "[XX]"
    else:
        icon = "[--]"
    print(f"  {icon} {name:28s} {result}")

print("-" * 60)
print(f"  PASS: {pass_count}  |  WARN: {warn_count}  |  FAIL: {fail_count}  |  NOT RUN: {not_run}")
print("=" * 60)

if fail_count == 0:
    print("\n  All components operational. Ready for live run.")
else:
    print(f"\n  {fail_count} component(s) failed. Review errors above.")